# Notebook 3: FLAN-T5 Fraud-Intelligence Generator

This notebook implements Phase 2b. A key dataset limitation must be stated clearly: the CFPB data
contains complaint narratives, but it does **not** contain analyst-written fraud intelligence
summaries. Therefore, a supervised FLAN-T5 experiment requires target summaries created through
human annotation or documented weak supervision.

The notebook supports both:

- **Human targets**, when a `human_summary` column is available
- **Weak template targets**, for pipeline testing and a preliminary baseline only

Final claims about generation quality should be based on a human-written test set that was never
used to construct the templates.


## 1. Install dependencies


In [ ]:
%pip install -q "transformers>=4.46,<6" "datasets>=3,<5" "evaluate>=0.4,<1" \
  "accelerate>=1,<2" "torch>=2.2" sentencepiece rouge-score bert-score pandas scikit-learn


## 2. Imports and configuration


In [ ]:
import inspect, json, random, re, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.red_flags import extract_red_flags, score_risk

with open(PROJECT_ROOT / "configs/project_config.json", encoding="utf-8") as f:
    CONFIG = json.load(f)

SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = PROJECT_ROOT / CONFIG["processed_dir"]
RESULTS_DIR = PROJECT_ROOT / "results" / "generator"
MODEL_DIR = PROJECT_ROOT / "models" / "generator"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = CONFIG["generator_model"]
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 96
FAST_DEV_RUN = True


## 3. Build structured signals and an annotation template


In [ ]:
train_df = pd.read_csv(DATA_DIR / "split_train.csv.gz")
val_df = pd.read_csv(DATA_DIR / "split_val.csv.gz")
test_df = pd.read_csv(DATA_DIR / "split_test.csv.gz")


def add_structured_signals(frame):
    frame = frame.copy()
    frame["red_flags"] = frame["clean_text"].map(extract_red_flags)
    risk = frame.apply(lambda row: score_risk(row["red_flags"], str(row["archetype_label"])), axis=1)
    frame["risk_level"] = [item[0] for item in risk]
    frame["risk_score"] = [item[1] for item in risk]
    if "cluster_confidence" not in frame:
        frame["cluster_confidence"] = np.nan
    return frame

train_df = add_structured_signals(train_df)
val_df = add_structured_signals(val_df)
test_df = add_structured_signals(test_df)

# Create a stratified annotation template with an explicit split column.
annotation_parts = []
for split_name, frame, per_class in [
    ("train", train_df, 30),
    ("val", val_df, 10),
    ("test", test_df, 10),
]:
    sampled = (
        frame.groupby("archetype_label", group_keys=False)
        .apply(lambda group: group.sample(min(len(group), per_class), random_state=SEED))
        .copy()
    )
    sampled["split"] = split_name
    annotation_parts.append(sampled)

annotation_template = pd.concat(annotation_parts, ignore_index=True)[[
    "split", "Complaint ID", "Consumer complaint narrative", "archetype_label",
    "red_flags", "risk_level"
]]
annotation_template["human_summary"] = ""
annotation_template["reviewer"] = ""
annotation_template["review_notes"] = ""
annotation_template.to_csv(DATA_DIR / "human_summary_annotation_template.csv", index=False)
print("Annotation rows:", len(annotation_template))

# After annotation, save the completed file as human_summaries.csv.
HUMAN_SUMMARIES_PATH = DATA_DIR / "human_summaries.csv"
if HUMAN_SUMMARIES_PATH.exists():
    human = pd.read_csv(HUMAN_SUMMARIES_PATH)
    required = {"Complaint ID", "human_summary"}
    missing = required.difference(human.columns)
    if missing:
        raise ValueError(f"Human summary file is missing columns: {sorted(missing)}")
    human = human[["Complaint ID", "human_summary"]].drop_duplicates("Complaint ID")
    for frame in (train_df, val_df, test_df):
        frame.drop(columns=["human_summary"], errors="ignore", inplace=True)
        frame["Complaint ID"] = frame["Complaint ID"].astype(str)
    human["Complaint ID"] = human["Complaint ID"].astype(str)
    train_df = train_df.merge(human, on="Complaint ID", how="left")
    val_df = val_df.merge(human, on="Complaint ID", how="left")
    test_df = test_df.merge(human, on="Complaint ID", how="left")
    print("Loaded completed human summaries:", human["human_summary"].notna().sum())
else:
    print("No completed human_summaries.csv found; the preliminary run will use weak targets.")


## 4. Create weak targets for a preliminary pipeline test

These targets are deterministic summaries of the already extracted fields. They are useful for
checking that tokenization, training, generation, and factual metrics work end to end. They are not
independent gold labels and should not be used as the sole evidence that the model discovered new
information.


In [ ]:
def format_flags(flags):
    if not flags:
        return "no explicit high-confidence red flag was extracted"
    if len(flags) == 1:
        return flags[0]
    return ", ".join(flags[:-1]) + ", and " + flags[-1]


def weak_target(row):
    return (
        f"{row['risk_level']} risk {row['archetype_label']} complaint. "
        f"The narrative shows {format_flags(row['red_flags'])}. "
        "The reported transaction and authorization timeline should be reviewed."
    )


def build_prompt(row):
    confidence = row.get("cluster_confidence")
    confidence_text = "not available" if pd.isna(confidence) else f"{float(confidence):.2f}"
    return (
        "Generate one concise, fact-grounded fraud intelligence summary. "
        "Use only the supplied fields and do not invent names, amounts, dates, or actions.\n"
        f"Complaint: {row['clean_text']}\n"
        f"Archetype: {row['archetype_label']}\n"
        f"Archetype confidence: {confidence_text}\n"
        f"Red flags: {format_flags(row['red_flags'])}\n"
        f"Risk level: {row['risk_level']}\n"
        "Summary:"
    )

for frame in (train_df, val_df, test_df):
    frame["input_text"] = frame.apply(build_prompt, axis=1)
    frame["weak_summary"] = frame.apply(weak_target, axis=1)
    if "human_summary" in frame.columns:
        human_text = frame["human_summary"].fillna("").astype(str)
        frame["target_text"] = frame["weak_summary"]
        frame.loc[human_text.str.strip().ne(""), "target_text"] = human_text[human_text.str.strip().ne("")]
        frame["target_source"] = np.where(human_text.str.strip().ne(""), "human", "weak")
    else:
        frame["target_text"] = frame["weak_summary"]
        frame["target_source"] = "weak"

if not FAST_DEV_RUN and (test_df["target_source"] != "human").any():
    raise ValueError("Final evaluation requires human summaries for every selected test record.")

train_df[["input_text", "target_text", "target_source", "archetype_label", "red_flags", "risk_level"]].head(3)


## 5. Zero-shot FLAN-T5 baseline


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model.to(device)

sample = test_df.sample(min(10, len(test_df)), random_state=SEED).copy()
encoded = tokenizer(
    sample["input_text"].tolist(),
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=MAX_INPUT_LENGTH,
).to(device)
with torch.no_grad():
    generated = base_model.generate(**encoded, max_new_tokens=MAX_TARGET_LENGTH, num_beams=4)
sample["zero_shot_summary"] = tokenizer.batch_decode(generated, skip_special_tokens=True)
sample[["archetype_label", "red_flags", "zero_shot_summary"]]


## 6. Fine-tune FLAN-T5


In [ ]:
import evaluate
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments


def stratified_cap(frame, cap_per_class):
    return (
        frame.groupby("archetype_label", group_keys=False)
        .apply(lambda group: group.sample(min(len(group), cap_per_class), random_state=SEED))
        .sample(frac=1.0, random_state=SEED)
        .reset_index(drop=True)
    )

if FAST_DEV_RUN:
    train_run = stratified_cap(train_df, 250)
    val_run = stratified_cap(val_df, 60)
    test_run = stratified_cap(test_df, 60)
    EPOCHS = 1
else:
    train_run, val_run, test_run = train_df, val_df, test_df
    EPOCHS = 3


def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["input_text"], truncation=True, max_length=MAX_INPUT_LENGTH
    )
    labels = tokenizer(
        text_target=batch["target_text"], truncation=True, max_length=MAX_TARGET_LENGTH
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


def to_dataset(frame):
    ds = Dataset.from_pandas(frame[["input_text", "target_text"]], preserve_index=False)
    return ds.map(tokenize_batch, batched=True, remove_columns=["input_text", "target_text"])

train_ds = to_dataset(train_run)
val_ds = to_dataset(val_run)
test_ds = to_dataset(test_run)

rouge = evaluate.load("rouge")

def compute_seq2seq_metrics(eval_prediction):
    predictions, labels_array = eval_prediction
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels_array = np.where(labels_array != -100, labels_array, tokenizer.pad_token_id)
    pred_text = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    label_text = tokenizer.batch_decode(labels_array, skip_special_tokens=True)
    values = rouge.compute(predictions=pred_text, references=label_text)
    return {key: float(value) for key, value in values.items()}

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
args_kwargs = dict(
    output_dir=str(MODEL_DIR),
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.10,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)
signature = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
args_kwargs["eval_strategy" if "eval_strategy" in signature else "evaluation_strategy"] = "epoch"
training_args = Seq2SeqTrainingArguments(**args_kwargs)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
    compute_metrics=compute_seq2seq_metrics,
)
trainer_signature = inspect.signature(Seq2SeqTrainer.__init__).parameters
if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)
trainer.train()
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)


## 7. Factual-consistency checks


In [ ]:
prediction = trainer.predict(test_ds)
pred_ids = prediction.predictions[0] if isinstance(prediction.predictions, tuple) else prediction.predictions
pred_text = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

evaluation = test_run.reset_index(drop=True).copy()
evaluation["generated_summary"] = pred_text

def normalized_contains(text, phrase):
    normalize = lambda value: re.sub(r"[^a-z0-9]+", " ", str(value).lower()).strip()
    return normalize(phrase) in normalize(text)

def flag_coverage(row):
    flags = row["red_flags"]
    if not flags:
        return np.nan
    hits = sum(normalized_contains(row["generated_summary"], flag) for flag in flags)
    return hits / len(flags)

evaluation["archetype_correct"] = evaluation.apply(
    lambda row: normalized_contains(row["generated_summary"], row["archetype_label"]), axis=1
)
evaluation["red_flag_coverage"] = evaluation.apply(flag_coverage, axis=1)
evaluation["risk_correct"] = evaluation.apply(
    lambda row: normalized_contains(row["generated_summary"], row["risk_level"]), axis=1
)

factual_metrics = {
    "archetype_mention_rate": float(evaluation["archetype_correct"].mean()),
    "mean_red_flag_coverage": float(evaluation["red_flag_coverage"].dropna().mean()),
    "risk_level_mention_rate": float(evaluation["risk_correct"].mean()),
}
print(factual_metrics)
evaluation.to_csv(RESULTS_DIR / "generator_predictions.csv", index=False)


## 8. BERTScore and human review


In [ ]:
# BERTScore is more expensive, so cap it for a preliminary run.
bertscore_sample = evaluation.head(min(500, len(evaluation)))
bertscore = evaluate.load("bertscore")
bert_values = bertscore.compute(
    predictions=bertscore_sample["generated_summary"].tolist(),
    references=bertscore_sample["target_text"].tolist(),
    lang="en",
)
print("Mean BERTScore F1:", float(np.mean(bert_values["f1"])))

human_review = evaluation[[
    "Complaint ID", "Consumer complaint narrative", "archetype_label", "red_flags",
    "risk_level", "target_text", "generated_summary"
]].sample(min(100, len(evaluation)), random_state=SEED)
human_review["clarity_1_to_5"] = ""
human_review["relevance_1_to_5"] = ""
human_review["factuality_1_to_5"] = ""
human_review["analyst_usefulness_1_to_5"] = ""
human_review["review_notes"] = ""
human_review.to_csv(RESULTS_DIR / "human_review_rubric.csv", index=False)
print("Saved generator outputs to", RESULTS_DIR)


## Final-generation requirements

Before making final claims:

1. Complete a human-written summary set with clear annotation guidelines.
2. Keep its test portion separate from all prompt/template design.
3. Compare zero-shot FLAN-T5 against fine-tuned FLAN-T5 on the same human test set.
4. Report BERTScore alongside archetype accuracy, red-flag coverage, hallucination review, and the
   human rubric. BERTScore alone cannot establish factual consistency.
